# SmallENet Results Preview -- Dataset502_ARCADE_6x6_1c

`SmallENet` (`lightm-unet/nnunetv2/nets/SmallENet.py`, trained via
`nnUNetTrainerSmallENet`) is a small, full-resolution, single-sigmoid-logit
binary segmenter for the `6x6`-tiled ARCADE vessel/background patches. Its
`perform_actual_validation()` deliberately raises `NotImplementedError` --
the network's 1-channel sigmoid output isn't wired up to nnU-Net's normal
`num_segmentation_heads`-channel sliding-window export path, so
`nnUNetv2_predict` cannot be used for this model. This notebook instead
loads the checkpoint directly and runs the network by hand: per-image
z-score normalization (matching `ZScoreNormalization`, `use_mask_for_norm
=False`, per `plans.json`), forward pass, `sigmoid > 0.5`.

Ground truth: `data/nnUNet_raw/Dataset502_ARCADE_6x6_1c/{imagesTs,labelsTs}`
(18,300 held-out patches). Classes: `0=background, 1=vessel`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import torch
from PIL import Image


def _find_repo_root(start: Path) -> Path:
    """Walk up from cwd looking for the repo root, so this notebook works the
    same whether Jupyter's cwd is the repo root or analysis/502_ARCADE_6x6_1c itself."""
    for candidate in [start, *start.parents]:
        if (candidate / "lightm-unet").exists() and (candidate / "data").exists():
            return candidate
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
PACKAGE_ROOT = REPO_ROOT / "lightm-unet"
sys.path.insert(0, str(PACKAGE_ROOT))
from nnunetv2.nets.SmallENet import SmallENet  # noqa: E402

DATASET_NAME = "Dataset503_ARCADE_6x6_1c_ve2to1"
TRAINER_NAME = "nnUNetTrainerSmallENet"
PLANS_NAME = "nnUNetPlans"
CONFIGURATION = "2d"
FOLD = 0
CHECKPOINT_NAME = "checkpoint_best.pth"

DATASET_DIR = REPO_ROOT / "data" / "nnUNet_raw" / DATASET_NAME
IMAGES_TS_DIR = DATASET_DIR / "imagesTs"
LABELS_TS_DIR = DATASET_DIR / "labelsTs"
NNUNET_RESULTS = REPO_ROOT / "data" / "nnUNET_results"
CHECKPOINT_PATH = (
    NNUNET_RESULTS / DATASET_NAME / f"{TRAINER_NAME}__{PLANS_NAME}__{CONFIGURATION}"
    / f"fold_{FOLD}" / CHECKPOINT_NAME
)
METRICS_DIR = REPO_ROOT / "analysis" / "502_ARCADE_6x6_1c" / "results"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# SmallENet's own hyperparameters (must match SMALLENET_* env vars used at
# training time -- train_small_enet_6x6_1c.job left all of these at default).
INITIAL_CHANNELS = 16
STAGE_CHANNELS = 32
LCN_KERNEL_SIZE = 9
THRESHOLD = 0.5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Repo root      :", REPO_ROOT)
print("Dataset dir    :", DATASET_DIR)
print("ImagesTs       :", IMAGES_TS_DIR)
print("LabelsTs       :", LABELS_TS_DIR)
print("Checkpoint     :", CHECKPOINT_PATH)
print("Device         :", DEVICE)

Repo root      : /workspace/LightM-UNet
Dataset dir    : /workspace/LightM-UNet/data/nnUNet_raw/Dataset503_ARCADE_6x6_1c_ve2to1
ImagesTs       : /workspace/LightM-UNet/data/nnUNet_raw/Dataset503_ARCADE_6x6_1c_ve2to1/imagesTs
LabelsTs       : /workspace/LightM-UNet/data/nnUNet_raw/Dataset503_ARCADE_6x6_1c_ve2to1/labelsTs
Checkpoint     : /workspace/LightM-UNet/data/nnUNET_results/Dataset503_ARCADE_6x6_1c_ve2to1/nnUNetTrainerSmallENet__nnUNetPlans__2d/fold_0/checkpoint_best.pth
Device         : cuda


## Load the model

Checkpoints save the raw `state_dict` under `network_weights`
(`nnUNetTrainer.save_checkpoint`); no nnU-Net trainer/plans machinery is
needed to load and run the network standalone.

In [9]:
model = SmallENet(
    in_channels=1,
    out_channels=1,
    initial_channels=INITIAL_CHANNELS,
    stage_channels=STAGE_CHANNELS,
    lcn_kernel_size=LCN_KERNEL_SIZE,
)

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["network_weights"])
model = model.to(DEVICE).eval()

print(f"Loaded {CHECKPOINT_NAME} from epoch {checkpoint.get('current_epoch')}")
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

Loaded checkpoint_best.pth from epoch 117
Parameters: 8,297


## Build the test-split sample index and inference helpers

Filenames follow `test_<case>_<p|b><patch_idx>[_0000].png`; match `imagesTs`
to `labelsTs` by stripping the nnU-Net `_0000` channel suffix.

In [10]:
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def case_id_from_image(path: Path) -> str:
    stem = path.stem
    return stem[:-5] if stem.endswith("_0000") else stem


def image_files(folder: Path) -> list[Path]:
    return sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS)


img_files = image_files(IMAGES_TS_DIR)
gt_files = {p.stem: p for p in image_files(LABELS_TS_DIR)}

samples = []
for img in img_files:
    cid = case_id_from_image(img)
    if cid in gt_files:
        samples.append({"case_id": cid, "image": img, "gt": gt_files[cid]})

print(f"ImagesTs found  : {len(img_files)}")
print(f"LabelsTs found  : {len(gt_files)}")
print(f"Matched samples : {len(samples)}")
if not samples:
    raise FileNotFoundError(f"No matched samples found under {DATASET_DIR}")


def load_gray(path: Path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("L"), dtype=np.float32)


def load_gt(path: Path) -> np.ndarray:
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    return arr


def zscore_normalize(image: np.ndarray) -> np.ndarray:
    """Matches nnunetv2.preprocessing.normalization.ZScoreNormalization with
    use_mask_for_norm=False (whole-image mean/std, per plans.json)."""
    mean = image.mean()
    std = image.std()
    return (image - mean) / max(std, 1e-8)


@torch.no_grad()
def predict_case(idx: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Returns (raw grayscale, GT binary mask, predicted probability map)."""
    item = samples[idx]
    raw = load_gray(item["image"])
    gt = (load_gt(item["gt"]) > 0)

    normed = zscore_normalize(raw)
    x = torch.from_numpy(normed).float()[None, None].to(DEVICE)  # 1,1,H,W
    logits = model(x)
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    return raw, gt, prob

AttributeError: 'str' object has no attribute 'iterdir'

## Preview: raw patch, ground truth, prediction, confusion overlay

In [ ]:
CLASS_COLORS = ["#000000", "#ffffff"]
CMAP = mcolors.ListedColormap(CLASS_COLORS)
NORM = mcolors.BoundaryNorm([0, 1, 2], 2)
IMSHOW_MASK_KW = dict(cmap=CMAP, norm=NORM, interpolation="nearest")

CONFUSION_COLORS = {
    "TP": np.array([0, 200, 0, 210], dtype=np.uint8),
    "FP": np.array([220, 0, 0, 200], dtype=np.uint8),
    "FN": np.array([255, 190, 0, 210], dtype=np.uint8),
}


def make_confusion(gt: np.ndarray, pred: np.ndarray) -> np.ndarray:
    overlay = np.zeros((*gt.shape, 4), dtype=np.uint8)
    overlay[gt & pred] = CONFUSION_COLORS["TP"]
    overlay[~gt & pred] = CONFUSION_COLORS["FP"]
    overlay[gt & ~pred] = CONFUSION_COLORS["FN"]
    return overlay


def show_confusion_grid(indices, save_path: Path | None = None):
    indices = list(indices)
    fig, axes = plt.subplots(len(indices), 4, figsize=(14, 3.4 * len(indices)))
    axes = np.atleast_2d(axes)
    for col, title in enumerate(["Raw", "Ground truth", "Prediction", "Confusion"]):
        axes[0, col].set_title(title, fontsize=11, fontweight="bold")
    for row, idx in enumerate(indices):
        raw, gt, prob = predict_case(idx)
        pred = prob > THRESHOLD
        confusion = make_confusion(gt, pred)
        axes[row, 0].imshow(raw, cmap="gray")
        axes[row, 0].set_ylabel(f"#{idx}\n{samples[idx]['case_id']}", fontsize=7, rotation=0, labelpad=32, va="center")
        axes[row, 0].axis("off")
        axes[row, 1].imshow(gt, **IMSHOW_MASK_KW)
        axes[row, 1].axis("off")
        axes[row, 2].imshow(pred, **IMSHOW_MASK_KW)
        axes[row, 2].axis("off")
        axes[row, 3].imshow(raw, cmap="gray")
        axes[row, 3].imshow(confusion)
        axes[row, 3].axis("off")
    legend = [
        mpatches.Patch(color=CONFUSION_COLORS["TP"] / 255, label="TP"),
        mpatches.Patch(color=CONFUSION_COLORS["FP"] / 255, label="FP"),
        mpatches.Patch(color=CONFUSION_COLORS["FN"] / 255, label="FN"),
    ]
    fig.legend(handles=legend, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.01), frameon=False)
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"Saved: {save_path}")
    plt.show()


# A mix of vessel-containing and background-only patches for a representative preview.
rng = np.random.default_rng(0)
preview_indices = rng.choice(len(samples), size=100, replace=False)
show_confusion_grid(sorted(preview_indices), save_path=METRICS_DIR / f"{TRAINER_NAME}_confusion_grid_display.png")

## Metrics over the full test split

Pooled dice/precision/recall/accuracy across all 18,300 patches, plus a
per-image table saved to `results/`.

In [ ]:
EPS = 1e-8


def prf_from_counts(tp: int, fp: int, fn: int, tn: int) -> dict[str, float]:
    precision = tp / (tp + fp + EPS)
    recall = tp / (tp + fn + EPS)
    dice = (2 * tp) / (2 * tp + fp + fn + EPS)
    accuracy = (tp + tn) / (tp + fp + fn + tn + EPS)
    return {"dice_f1": dice, "precision": precision, "recall_sensitivity": recall, "accuracy": accuracy}


rows = []
pooled = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}

for idx, item in enumerate(samples):
    raw, gt, prob = predict_case(idx)
    pred = prob > THRESHOLD
    tp = int((gt & pred).sum())
    fp = int((~gt & pred).sum())
    fn = int((gt & ~pred).sum())
    tn = int((~gt & ~pred).sum())
    pooled["tp"] += tp; pooled["fp"] += fp; pooled["fn"] += fn; pooled["tn"] += tn
    metrics = prf_from_counts(tp, fp, fn, tn)
    rows.append({"case_id": item["case_id"], "gt_vessel_px": int(gt.sum()), **metrics})

per_image_metrics = pd.DataFrame(rows)
overall_metrics = pd.DataFrame([{**prf_from_counts(**pooled), "n_images": len(samples)}])

print("Overall metrics")
display(overall_metrics)
print("Per-image metrics (head)")
display(per_image_metrics.head())

per_image_metrics.to_csv(METRICS_DIR / f"{TRAINER_NAME}_per_image_metrics.csv", index=False)
overall_metrics.to_csv(METRICS_DIR / f"{TRAINER_NAME}_overall_metrics.csv", index=False)
print("Saved metrics to", METRICS_DIR)

## Per-patch dice distribution

Empty-GT patches (no vessel pixels) score dice=0 by the standard formula
even on a perfect all-background prediction, so they're split out and
reported separately rather than pooled into the same histogram.

In [ ]:
empty_gt = per_image_metrics["gt_vessel_px"] == 0
non_empty = per_image_metrics[~empty_gt]
print(f"Empty-GT patches: {int(empty_gt.sum())} / {len(per_image_metrics)}")
if len(non_empty):
    print(f"Vessel-containing patches -- mean dice: {non_empty['dice_f1'].mean():.4f}, "
          f"median dice: {non_empty['dice_f1'].median():.4f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(non_empty["dice_f1"], bins=40, color="#4C72B0", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Per-patch dice (vessel-containing GT patches only)")
ax.set_ylabel("Number of patches")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.25)
fig.suptitle(f"{TRAINER_NAME}: per-patch dice distribution", fontweight="bold")
plt.tight_layout()
fig.savefig(METRICS_DIR / f"{TRAINER_NAME}_per_patch_dice_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

## Multi-scale reconstruction: 6x6 + 5x5 patch fusion

Each test-split patch is scored in isolation above; this section reassembles
whole source frames from their patches to see how per-patch predictions
combine at the image level, and compares a few fusion strategies:

- **6x6 only** -- the non-overlapping layer-1 grid (`p{row}{col}` patches)
  tiled back together. This alone already covers the full image.
- **5x5 only** -- the border-straddling layer-2 grid (`b{row}{col}` patches),
  each centered on an internal layer-1 grid vertex (`dataset-prep/patch_grid.py`).
  No layer-2 patch reaches the image border, so this view has an uncovered
  border ring, shown as background/0.
- **OR (6x6 | 5x5)** -- pixelwise logical OR of the two thresholded views.
- **Confidence blend** -- see below.

In [ ]:
import re

sys.path.insert(0, str(REPO_ROOT / "dataset-prep"))
from patch_grid import layer1_boxes, layer2_boxes  # noqa: E402

GRID_COLS, GRID_ROWS = 6, 6  # matches Dataset502_ARCADE_6x6_1c's 6x6 grid

FNAME_RE = re.compile(r"^(?P<stem>.+)_(?P<kind>[pb])(?P<row>\d{2})(?P<col>\d{2})$")


def parse_case_id(case_id: str):
    """'test_100_p0502' -> ('test_100', 'p', 5, 2); None if it doesn't match
    the grid-patch naming convention at all."""
    m = FNAME_RE.match(case_id)
    if not m:
        return None
    return m.group("stem"), m.group("kind"), int(m.group("row")), int(m.group("col"))


case_by_id = {s["case_id"]: s for s in samples}
case_id_to_idx = {s["case_id"]: i for i, s in enumerate(samples)}

# Group every test patch by its source frame ("stem") and grid position.
stems: dict[str, dict[str, dict[tuple[int, int], str]]] = {}
for s in samples:
    parsed = parse_case_id(s["case_id"])
    if parsed is None:
        continue
    stem, kind, row, col = parsed
    stems.setdefault(stem, {"p": {}, "b": {}})[kind][(row, col)] = s["case_id"]

complete_stems = sorted(
    stem for stem, entry in stems.items()
    if len(entry["p"]) == GRID_ROWS * GRID_COLS and len(entry["b"]) == (GRID_ROWS - 1) * (GRID_COLS - 1)
)
print(f"Source images found         : {len(stems)}")
print(f"With full 6x6 + 5x5 patches : {len(complete_stems)}")

## Confidence ring: per-pixel blend weight within a patch

For the confidence blend, every patch contributes a per-pixel weight instead
of an all-or-nothing vote: the innermost region (`center_frac` of the patch's
pixel area, default 20%) gets `max_confidence` (default 1.0, i.e. fully
trusted), smoothly falling off to `min_confidence` (default 0.5) at the
patch's outer edge/corner. Where two patches overlap (any pixel under a 5x5
patch is always also covered by up to four 6x6 patches), the final
probability is a weighted average of every patch's prediction at that pixel,
weighted by each patch's own confidence there -- so a pixel near the center of
one patch but near the ragged edge of another leans toward the more confident
(central) one.

`metric` controls the ring shape: `"chebyshev"` (default) gives nested
*squares* -- matches the patches' own shape -- `"euclidean"` gives nested
circles. `falloff` controls the shape of the transition from `max_confidence`
to `min_confidence`: `"linear"` or `"gaussian"`.

In [ ]:
def confidence_weight_map(
    shape: tuple[int, int],
    center_frac: float = 0.2,
    min_confidence: float = 0.5,
    max_confidence: float = 1.0,
    falloff: str = "linear",
    metric: str = "chebyshev",
) -> np.ndarray:
    """Per-pixel blend weight for one patch of the given (H, W) shape:
    max_confidence within an innermost region covering center_frac of the
    patch's area, smoothly decreasing to min_confidence at the outer edge."""
    h, w = shape
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float64)
    cy, cx = (h - 1) / 2.0, (w - 1) / 2.0
    dy, dx = np.abs(yy - cy), np.abs(xx - cx)

    if metric == "chebyshev":
        dist = np.maximum(dy, dx)
        max_dist = max(cy, cx, 1e-8)
    elif metric == "euclidean":
        dist = np.sqrt(dy ** 2 + dx ** 2)
        max_dist = np.sqrt(cy ** 2 + cx ** 2) + 1e-8
    else:
        raise ValueError(f"Unknown metric: {metric!r} (expected 'chebyshev' or 'euclidean')")
    d = np.clip(dist / max_dist, 0.0, 1.0)  # 0 at center, 1 at outer edge/corner

    # Boundary (in normalized d) enclosing center_frac of the patch's area:
    # for nested squares the area fraction inside radius d0 is d0**2; for
    # nested circles it's the same to a very good approximation as long as
    # the circle stays inside the inscribed circle (true for any
    # center_frac <~ 0.785, well beyond the 0.2 default).
    d0 = np.sqrt(np.clip(center_frac, 0.0, 1.0))

    t = np.clip((d - d0) / max(1.0 - d0, 1e-8), 0.0, 1.0)  # 0 at d0, 1 at the edge
    if falloff == "linear":
        curve = t
    elif falloff == "gaussian":
        curve = 1.0 - np.exp(-3.0 * t ** 2)
        curve /= (1.0 - np.exp(-3.0))  # renormalize so curve(1) == 1 exactly
    else:
        raise ValueError(f"Unknown falloff: {falloff!r} (expected 'linear' or 'gaussian')")

    weight = max_confidence - curve * (max_confidence - min_confidence)
    return weight.astype(np.float32)


RING_KWARGS = dict(center_frac=0.2, min_confidence=0.5, max_confidence=1.0, falloff="linear", metric="chebyshev")

# Sanity-check: what the ring actually looks like on one patch, both falloffs.
_shape = load_gray(samples[0]["image"]).shape
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, falloff in zip(axes[:2], ["linear", "gaussian"]):
    w = confidence_weight_map(_shape, **{**RING_KWARGS, "falloff": falloff})
    im = ax.imshow(w, cmap="viridis", vmin=RING_KWARGS["min_confidence"], vmax=RING_KWARGS["max_confidence"])
    ax.set_title(f"weight map (falloff={falloff})", fontsize=10)
    ax.axis("off")
fig.colorbar(im, ax=axes[1], fraction=0.046, label="confidence weight")
axes[2].plot(confidence_weight_map(_shape, **RING_KWARGS)[_shape[0] // 2, :], label="linear")
axes[2].plot(confidence_weight_map(_shape, **{**RING_KWARGS, "falloff": "gaussian"})[_shape[0] // 2, :], label="gaussian")
axes[2].set_title("weight along horizontal center line", fontsize=10)
axes[2].set_xlabel("x (px)"); axes[2].set_ylabel("confidence weight")
axes[2].set_ylim(0, 1.05)
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Reconstruct + blend one source image

`load_source_patches()` runs (cached) inference on every 6x6 + 5x5 patch of
one source frame and places them on full-size canvases -- independent of the
confidence-ring parameters. `blend_source()` does the actual per-pixel
weighted blend and is cheap (no network inference), so it's safe to re-run
repeatedly while tuning `RING_KWARGS`.

In [ ]:
_patch_cache: dict[str, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}


def predict_cached(case_id: str):
    if case_id not in _patch_cache:
        _patch_cache[case_id] = predict_case(case_id_to_idx[case_id])
    return _patch_cache[case_id]


def load_source_patches(stem: str) -> dict:
    """Run inference on every patch of one source image and place raw/GT/
    probability on full-size canvases. Also keeps each patch's (box, prob) so
    blend_source() can weight-blend them without redoing any network work."""
    entry = stems[stem]
    p_ids, b_ids = entry["p"], entry["b"]
    if len(p_ids) != GRID_ROWS * GRID_COLS:
        raise ValueError(f"{stem}: expected {GRID_ROWS * GRID_COLS} p-patches, got {len(p_ids)}")

    # Full canvas size from the p-patches' own pixel dimensions -- layer1_boxes
    # tiles the image exactly (no gaps/overlap), so row-0 widths sum to the
    # full width and column-0 heights sum to the full height.
    row0_widths = [load_gray(case_by_id[p_ids[(0, c)]]["image"]).shape[1] for c in range(GRID_COLS)]
    col0_heights = [load_gray(case_by_id[p_ids[(r, 0)]]["image"]).shape[0] for r in range(GRID_ROWS)]
    width, height = sum(row0_widths), sum(col0_heights)

    boxes1 = layer1_boxes(width, height, GRID_COLS, GRID_ROWS)
    boxes2 = layer2_boxes(width, height, GRID_COLS, GRID_ROWS)

    raw_canvas = np.zeros((height, width), dtype=np.float32)
    gt_canvas = np.zeros((height, width), dtype=bool)
    prob1_canvas = np.zeros((height, width), dtype=np.float32)
    prob2_canvas = np.zeros((height, width), dtype=np.float32)
    prob2_covered = np.zeros((height, width), dtype=bool)
    p_patches, b_patches = [], []

    for r in range(GRID_ROWS):
        for c in range(GRID_COLS):
            box = boxes1[r][c]
            x0, y0, x1, y1 = box
            raw, gt, prob = predict_cached(p_ids[(r, c)])
            raw_canvas[y0:y1, x0:x1] = raw
            gt_canvas[y0:y1, x0:x1] = gt
            prob1_canvas[y0:y1, x0:x1] = prob
            p_patches.append((box, prob))

    for r in range(GRID_ROWS - 1):
        for c in range(GRID_COLS - 1):
            case_id = b_ids.get((r, c))
            if case_id is None:
                continue
            box = boxes2[r][c]
            x0, y0, x1, y1 = box
            _, _, prob = predict_cached(case_id)
            prob2_canvas[y0:y1, x0:x1] = prob
            prob2_covered[y0:y1, x0:x1] = True
            b_patches.append((box, prob))

    return {
        "stem": stem, "width": width, "height": height,
        "raw": raw_canvas, "gt": gt_canvas,
        "prob_6x6": prob1_canvas, "prob_5x5": prob2_canvas, "prob_5x5_covered": prob2_covered,
        "p_patches": p_patches, "b_patches": b_patches,
    }


def blend_source(loaded: dict, ring_kwargs: dict | None = None) -> dict:
    """Confidence-ring-weighted blend of every patch covering each pixel.
    No network inference here -- cheap to call repeatedly while tuning ring_kwargs."""
    ring_kwargs = ring_kwargs or {}
    height, width = loaded["height"], loaded["width"]
    weighted_sum = np.zeros((height, width), dtype=np.float32)
    weight_sum = np.zeros((height, width), dtype=np.float32)

    for box, prob in loaded["p_patches"] + loaded["b_patches"]:
        x0, y0, x1, y1 = box
        w = confidence_weight_map(prob.shape, **ring_kwargs)
        weighted_sum[y0:y1, x0:x1] += prob * w
        weight_sum[y0:y1, x0:x1] += w

    blended_prob = weighted_sum / np.maximum(weight_sum, 1e-8)
    pred_6x6 = loaded["prob_6x6"] > THRESHOLD
    pred_5x5 = loaded["prob_5x5"] > THRESHOLD
    return {
        "pred_6x6": pred_6x6,
        "pred_5x5": pred_5x5,
        "pred_or": pred_6x6 | pred_5x5,
        "blended_prob": blended_prob,
        "pred_blend": blended_prob > THRESHOLD,
    }


def dice_for_views(gt: np.ndarray, blend: dict) -> pd.DataFrame:
    views = {
        "6x6 only": blend["pred_6x6"],
        "5x5 only": blend["pred_5x5"],
        "OR (6x6 | 5x5)": blend["pred_or"],
        "confidence blend": blend["pred_blend"],
    }
    rows = []
    for name, pred in views.items():
        tp = int((gt & pred).sum()); fp = int((~gt & pred).sum())
        fn = int((gt & ~pred).sum()); tn = int((~gt & ~pred).sum())
        rows.append({"view": name, **prf_from_counts(tp, fp, fn, tn)})
    return pd.DataFrame(rows)

## Preview a few reconstructed source images

Raw / GT / 6x6-only / 5x5-only / OR / confidence-blend, plus the confidence
blend's continuous probability map (where the ring's grey falloff is actually
visible -- the thresholded views next to it are hard masks) and a confusion
overlay of the final blended prediction on the raw image.

In [ ]:
def show_reconstruction(stem: str, ring_kwargs: dict | None = None, save_path: Path | None = None):
    loaded = load_source_patches(stem)
    blend = blend_source(loaded, ring_kwargs)
    gt = loaded["gt"]
    confusion_blend = make_confusion(gt, blend["pred_blend"])

    panels = [
        ("Raw (full)", loaded["raw"], dict(cmap="gray")),
        ("GT (full)", gt, IMSHOW_MASK_KW),
        ("6x6 only", blend["pred_6x6"], IMSHOW_MASK_KW),
        ("5x5 only", blend["pred_5x5"], IMSHOW_MASK_KW),
        ("OR (6x6 | 5x5)", blend["pred_or"], IMSHOW_MASK_KW),
        ("Confidence blend (prob)", blend["blended_prob"], dict(cmap="gray", vmin=0, vmax=1)),
        ("Confidence blend (thresholded)", blend["pred_blend"], IMSHOW_MASK_KW),
    ]
    fig, axes = plt.subplots(1, len(panels) + 1, figsize=(3.0 * (len(panels) + 1), 3.6))
    for ax, (title, arr, kw) in zip(axes, panels):
        ax.imshow(arr, **kw)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    axes[-1].imshow(loaded["raw"], cmap="gray")
    axes[-1].imshow(confusion_blend)
    axes[-1].set_title("Blend vs GT (raw overlay)", fontsize=9)
    axes[-1].axis("off")
    fig.suptitle(f"{stem}: multi-scale reconstruction", fontsize=12, fontweight="bold")
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", dpi=150)
        print(f"Saved: {save_path}")
    plt.show()

    dice_table = dice_for_views(gt, blend)
    print(f"Per-view dice for {stem}:")
    display(dice_table)
    return loaded, blend, dice_table


demo_stems = complete_stems[:3]
for _stem in demo_stems:
    show_reconstruction(_stem, RING_KWARGS, save_path=METRICS_DIR / f"{TRAINER_NAME}_{_stem}_multiscale_reconstruction.png")

## Dice on all views, aggregated over every reconstructed test image

Same four views (6x6 only / 5x5 only / OR / confidence blend), pooled per
image then averaged across every source frame with full 6x6+5x5 coverage.

In [ ]:
_agg_rows = []
for _stem in complete_stems:
    _loaded = load_source_patches(_stem)
    _blend = blend_source(_loaded, RING_KWARGS)
    _dice_table = dice_for_views(_loaded["gt"], _blend)
    _dice_table.insert(0, "stem", _stem)
    _agg_rows.append(_dice_table)

multiscale_metrics = pd.concat(_agg_rows, ignore_index=True)
multiscale_summary = (
    multiscale_metrics.groupby("view")[["dice_f1", "precision", "recall_sensitivity", "accuracy"]]
    .mean()
    .reset_index()
    .sort_values("dice_f1", ascending=False, ignore_index=True)
)

print(f"Multi-scale reconstruction dice, averaged over {len(complete_stems)} full test images:")
display(multiscale_summary)

multiscale_metrics.to_csv(METRICS_DIR / f"{TRAINER_NAME}_multiscale_per_image_metrics.csv", index=False)
multiscale_summary.to_csv(METRICS_DIR / f"{TRAINER_NAME}_multiscale_summary_metrics.csv", index=False)
print("Saved multi-scale metrics to", METRICS_DIR)